In [1]:
import cupy as cp

In [60]:
import numba

In [2]:
import pycuda

In [3]:
import torch

In [4]:
if torch.cuda.is_available():
    device_properties = torch.cuda.get_device_properties(0)
    print(f"Device Name: {device_properties.name}")
    print(f"Total Memory: {device_properties.total_memory / (1024**3):.2f} GB")
    print(f"CUDA Capability: {device_properties.major}.{device_properties.minor}")
else:
    print("CUDA is not available.")

Device Name: NVIDIA GeForce RTX 4090
Total Memory: 23.99 GB
CUDA Capability: 8.9


In [5]:
from minio import Minio
import os
import pandas as pd
import platform
import time

In [9]:
os.name

'posix'

In [10]:
print(platform.system(), platform.release())

Linux 6.6.87.2-microsoft-standard-WSL2


In [2]:
os.environ["SSL_CERT_FILE"] = 'all-data/public.crt'

In [3]:
def download_data_without_creds(data):
    client = Minio(
    '94.124.179.195:9000',
    secure=True
    )
    data = client.fget_object(
        'datasets',
        f'data/{data}',
        f'all-data/{data}'
    )

In [6]:
download_data_without_creds('OpenML-har-orig-1.csv')

In [7]:
download_data_without_creds('OpenML-usps-orig-1.csv')

In [11]:
# GaMAC Test

In [8]:
from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

In [11]:
def cupy_test(data, target_measures):
    measures = {"BR": Internal.BR, "OS": Internal.OS, "MCR": Internal.MCR, "SYM": Internal.SYM}
    used_measures = [measures[x] for x in target_measures.split(sep=',')]
    df = pd.read_csv(f'all-data/{data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
        # classes = [int(c[1]) for c in classes]
    else:
        classes = None
    df = df.drop('class', errors='ignore', axis=1)
    print(f'used data: {data}')
    print(f'used measures: {used_measures}')
    result = Gamac(target_measures=tuple(used_measures)).run(table=df, text=None, image=None, classes=classes)
    print(f'optimal.model: {result.model}')
    print(f'clusters: {result.model.labels_}')
    print(f'F1 score: {result.f1_score}')

In [17]:
start = time.time()
cupy_test('OpenML-har-orig.csv', 'SYM')
print('Work time:', time.time() - start)

used data: OpenML-har-orig.csv
used measures: [<Internal.SYM: ('sym', <function sym at 0x7f0017459c60>)>]


/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.44759058952331543s, {'SYM': 2.3566619348116e-07} ===
=== MEASURES 0.33731627464294434s, {'SYM': 0.0003814715373599406} ===
=== ALGO 2.6456451416015625s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 15, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 38.4997353553772s, FailedRun, {'bandwidth': 0.4505824806907181, 'max_iter': 294, 'tol': 2.6112746081016713e-05} ===
=== ALGO 5.558532238006592s, FailedRun, {'name': 'DBSCAN', 'eps': 0.6907286646112676, 'eps_sq': 0.47710608811566496, 'min_samples': 10} ===
=== ALGO 21.367307424545288s, FailedRun, {'min_cluster_size': 8, 'min_samples': 10} ===
=== MEASURES 0.24439668655395508s, {'SYM': 5.385651731288114e-05} ===
=== ALGO 36.13166308403015s, SuccessRun, {'threshold': 0.6952076084171146, 'branching_factor': 32, 'n_clusters': 7} ===
=== MEASURES 0.23161649703979492s, {'SYM': 0.0001628135804306658} ===
=== ALGO 0.3363487720489502s, SuccessRun, {'name': 'KMeans', 'n_clusters': 7, 'max_iter': 100, 'tol': 0.000

In [16]:
start = time.time()
cupy_test('OpenML-usps-orig.csv', 'OS')
print('Work time:', time.time() - start)

used data: OpenML-usps-orig.csv
used measures: [<Internal.OS: ('os', <function os at 0x7f001745b600>)>]


/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.004419565200805664s, {'OS': -11874.843649351544} ===
=== MEASURES 0.002147197723388672s, {'OS': -145.42566805747447} ===
=== ALGO 0.226912260055542s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 4, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 28.61929154396057s, FailedRun, {'bandwidth': 0.48027685436394224, 'max_iter': 104, 'tol': 4.513097774718048e-05} ===
=== ALGO 4.8600404262542725s, FailedRun, {'name': 'DBSCAN', 'eps': 0.35034741100116923, 'eps_sq': 0.1227433083952222, 'min_samples': 9} ===
=== ALGO 24.688478708267212s, FailedRun, {'min_cluster_size': 12, 'min_samples': 13} ===
=== MEASURES 0.010626554489135742s, {'OS': -56.866838101566266} ===
=== ALGO 62.66710925102234s, SuccessRun, {'threshold': 0.2228057754309866, 'branching_factor': 73, 'n_clusters': 9} ===
=== MEASURES 0.004667520523071289s, {'OS': -42.439932119747056} ===
=== ALGO 0.4781327247619629s, SuccessRun, {'name': 'KMeans', 'n_clusters': 12, 'max_iter': 100, 'tol': 0.0001, 'r

In [12]:
# CuPy test

In [44]:
from gamac.estimation.functions import f1

In [30]:
def euclidean_distance(x, y):
    return cp.linalg.norm(x - y)

In [42]:
class KMeansCuPy:
    def __init__(self, n_clusters=8, max_iter=300, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
    
    def initialize_centroids(self, X):
        idx = cp.random.choice(X.shape[0], size=self.n_clusters, replace=False)
        centroids = X[idx]
        return centroids
        
    def assign_labels(self, X, centroids):
        labels = []
        for i in range(len(X)):
            distances = [euclidean_distance(X[i], centroid) for centroid in centroids]
            distances = cp.asarray([float(x) for x in distances])
            label = cp.argmin(distances)
            labels.append(label)
        return cp.array(labels)
            
    def update_centroids(self, X, labels):
        new_centroids = []
        for c in range(self.n_clusters):
            points_in_cluster = X[labels == c]
            if len(points_in_cluster) > 0:
                mean_point = cp.mean(points_in_cluster, axis=0)
                new_centroids.append(mean_point)
            else:
                random_idx = cp.random.randint(0, len(X))
                new_centroids.append(X[random_idx])
        return cp.stack(new_centroids)
                
    def fit_predict(self, X):
        centroids = self.initialize_centroids(X)
        prev_centroids = None
        iteration = 0
        
        while True:
            labels = self.assign_labels(X, centroids)
            prev_centroids = centroids.copy()
            centroids = self.update_centroids(X, labels)
            iteration += 1
            
            if iteration >= self.max_iter or cp.all(cp.abs(centroids - prev_centroids) <= self.tol):
                break
        
        return labels

In [52]:
def test_kmeans_cupy(data):
    df = pd.read_csv(f'all-data/{data}')
    data_gpu = cp.asarray(df)
    print(f'used data: {data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
    else:
        classes = None
    model = KMeansCuPy(n_clusters=8, max_iter=100, tol=0.0001)
    predicted_labels = model.fit_predict(data_gpu)
    predicted_labels_gpu = cp.array(predicted_labels, dtype=cp.int32)
    classes_gpu =  cp.array(classes, dtype=cp.int32)
    f1_score = f1(classes_gpu, predicted_labels_gpu)
    print(f'clusters: {predicted_labels}')
    print(f'F1 score: {f1_score}')

In [54]:
start = time.time()
test_kmeans_cupy('OpenML-har-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
clusters: [1 1 1 ... 6 6 6]
F1 score: 0.6395327342094951
Work time: 777.075740814209


In [56]:
start = time.time()
test_kmeans_cupy('OpenML-usps-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-usps-orig-1.csv
clusters: [1 4 3 ... 3 7 5]
F1 score: 0.8200033042656077
Work time: 280.51096200942993


In [55]:
class MeanShiftCuPy:
    def __init__(self, bandwidth=None, min_bin_freq=1, max_iter=300):
        self.bandwidth = bandwidth
        self.min_bin_freq = min_bin_freq
        self.max_iter = max_iter
        self.cluster_centers_ = None
    
    def estimate_bandwidth(self, X, quantile=0.3):
        distances = cp.zeros((X.shape[0], X.shape[0]))
        for i in range(X.shape[0]):
            dist_i = cp.sqrt(cp.sum((X[i] - X)**2, axis=1))  # Евклидово расстояние
            distances[i] = dist_i
        sorted_distances = cp.sort(distances.flatten())[:len(distances)]
        bandwidth = cp.percentile(sorted_distances, q=(quantile * 100)).item() + 1e-7
        return bandwidth
    
    def shift_points(self, X):
        shifted_X = cp.empty_like(X)
        for i in range(X.shape[0]):
            neighbors_mask = cp.linalg.norm(X - X[i], axis=1) <= self.bandwidth
            neighbors = X[neighbors_mask]
            shifted_X[i] = cp.mean(neighbors, axis=0)
        return shifted_X
    
    def fit(self, X):
        if self.bandwidth is None:
            self.bandwidth = self.estimate_bandwidth(X)
        
        iterations = 0
        old_shifted_X = X.copy()
        while iterations < self.max_iter:
            shifted_X = self.shift_points(old_shifted_X)
            diff = cp.linalg.norm(shifted_X - old_shifted_X, axis=1)
            if cp.all(diff < 1e-3):
                break
            old_shifted_X = shifted_X
            iterations += 1
        
        unique_rows, indices = cp.unique(shifted_X, axis=0, return_inverse=True)
        freq = cp.bincount(indices)
        valid_indices = freq >= self.min_bin_freq
        self.cluster_centers_ = unique_rows[valid_indices].astype(float)
        return self
    
    def predict(self, X):
        if self.cluster_centers_ is None:
            raise ValueError("Model not fitted yet.")
        
        predictions = []
        for x in X:
            distances = cp.linalg.norm(self.cluster_centers_ - x, axis=1)
            closest_center_idx = cp.argmin(distances)
            predictions.append(closest_center_idx)
        return cp.array(predictions)

In [57]:
def test_meanshift_cupy(data):
    df = pd.read_csv(f'all-data/{data}')
    data_gpu = cp.asarray(df)
    print(f'used data: {data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
    else:
        classes = None
    model = MeanShiftCuPy(bandwidth=1.5)
    model.fit(data_gpu)
    predicted_labels = model.predict(data_gpu)
    predicted_labels_gpu = cp.array(predicted_labels, dtype=cp.int32)
    classes_gpu =  cp.array(classes, dtype=cp.int32)
    f1_score = f1(classes_gpu, predicted_labels_gpu)
    print(f'clusters: {predicted_labels}')
    print(f'F1 score: {f1_score}')

In [58]:
start = time.time()
test_meanshift_cupy('OpenML-har-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
clusters: [7741 5814 6362 ... 9521 1568  309]
F1 score: 0.0011657353190153927
Work time: 76.99848651885986


In [59]:
start = time.time()
test_meanshift_cupy('OpenML-usps-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-usps-orig-1.csv
clusters: [5412 4863 3596 ... 4115 1343 1563]
F1 score: 0.03998488030723101
Work time: 172.06875824928284


In [62]:
# numba test

In [71]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from numba import cuda, float32, int32
import math

@cuda.jit(device=True)
def squared_euclidean_distance(a, b):
    total = 0.0
    for i in range(a.shape[0]):
        diff = a[i] - b[i]
        total += diff * diff
    return total

@cuda.jit
def find_closest_centroid(data, centroids, output):
    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bw = cuda.blockDim.x
    pos = tx + bx * bw
    
    if pos < data.shape[0]:
        min_dist = float32(np.inf)
        assigned_centroid = -1
        for i in range(centroids.shape[0]):
            dist = squared_euclidean_distance(data[pos], centroids[i])
            if dist < min_dist:
                min_dist = dist
                assigned_centroid = i
        output[pos] = assigned_centroid

@cuda.jit
def calculate_new_centroids(data, labels, centroids):
    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bw = cuda.blockDim.x
    pos = tx + bx * bw
    
    if pos < centroids.shape[0]:  
        for feature in range(centroids.shape[1]):
            numerator = 0.0
            denominator = 0
            for point in range(data.shape[0]):
                if labels[point] == pos:
                    numerator += data[point][feature]
                    denominator += 1
            if denominator > 0:
                centroids[pos][feature] = numerator / denominator


def preprocess_data(df):
    numeric_cols = df.select_dtypes(include=['float', 'integer']).columns
    categorical_cols = df.select_dtypes(exclude=['float', 'integer']).columns
    
    scaler = StandardScaler()
    scaled_numeric_data = scaler.fit_transform(df[numeric_cols])
    
    encoder = OneHotEncoder(sparse_output=False)
    encoded_categorical_data = encoder.fit_transform(df[categorical_cols])
    
    preprocessed_data = np.hstack([scaled_numeric_data, encoded_categorical_data]).astype(np.float32)
    return preprocessed_data


def kmeans(data, k, max_iterations=300, tol=1e-4):
    centroids = data[np.random.choice(data.shape[0], k, replace=False)]
    previous_centroids = np.zeros_like(centroids)
    
    device_data = cuda.to_device(data)
    device_centroids = cuda.to_device(centroids.astype(np.float32))
    device_labels = cuda.device_array(shape=data.shape[0], dtype=np.int32)
    
    threads_per_block = 256
    blocks_per_grid = math.ceil(data.shape[0] / threads_per_block)
    
    for iter_num in range(max_iterations):
        find_closest_centroid[blocks_per_grid, threads_per_block](device_data, device_centroids, device_labels)
        calculate_new_centroids[blocks_per_grid, threads_per_block](device_data, device_labels, device_centroids)
        cuda.synchronize()
        current_centroids = device_centroids.copy_to_host()
        if np.allclose(current_centroids, previous_centroids, atol=tol):
            break
        previous_centroids = current_centroids.copy()
    
    final_labels = device_labels.copy_to_host()
    return final_labels, current_centroids

In [73]:
def test_kmeans_numba(data):
    df = pd.read_csv(f'all-data/{data}')
    data_gpu = cp.asarray(df)
    print(f'used data: {data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
    else:
        classes = None
    processed_data = preprocess_data(df)
    predicted_labels, centroids = kmeans(processed_data, k=8)
    predicted_labels_gpu = cp.array(predicted_labels, dtype=cp.int32)
    classes_gpu =  cp.array(classes, dtype=cp.int32)
    f1_score = f1(classes_gpu, predicted_labels_gpu)
    print(f'clusters: {predicted_labels}')
    print(f'F1 score: {f1_score}')

In [74]:
start = time.time()
test_kmeans_numba('OpenML-har-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
clusters: [6 6 6 ... 2 2 2]
F1 score: 0.5641885113869286
Work time: 6.850300073623657


In [75]:
start = time.time()
test_kmeans_numba('OpenML-usps-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-usps-orig-1.csv


/.venv/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 37 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


clusters: [7 2 4 ... 4 1 5]
F1 score: 0.6358633244915606
Work time: 10.081778287887573


In [84]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from numba import cuda, float32, int32
import math

@cuda.jit(device=True)
def squared_euclidean_distance(a, b):
    total = 0.0
    for i in range(a.shape[0]):
        diff = a[i] - b[i]
        total += diff * diff
    return total


@cuda.jit
def move_points(data, bandwith_squared, output):
    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bw = cuda.blockDim.x
    pos = tx + bx * bw
    
    if pos < data.shape[0]:
        D = data.shape[1]
        numerator = cuda.local.array(D, dtype=float32)
        denominator = 0.0
        
        for i in range(D):
            numerator[i] = 0.0
        
        for other_pos in range(data.shape[0]):
            dist = squared_euclidean_distance(data[pos], data[other_pos])
            weight = math.exp(-dist / bandwith_squared)
            denominator += weight
            for f in range(D):
                numerator[f] += weight * data[other_pos][f]
        
        if denominator > 0:
            for f in range(D):
                output[pos][f] = numerator[f] / denominator
        else:
            for f in range(D):
                output[pos][f] = data[pos][f]

def preprocess_data(df):
    numeric_cols = df.select_dtypes(include=['float', 'integer']).columns
    categorical_cols = df.select_dtypes(exclude=['float', 'integer']).columns
    
    scaler = StandardScaler()
    scaled_numeric_data = scaler.fit_transform(df[numeric_cols])
    
    encoder = OneHotEncoder(sparse_output=False)
    encoded_categorical_data = encoder.fit_transform(df[categorical_cols])
    
    preprocessed_data = np.hstack([scaled_numeric_data, encoded_categorical_data]).astype(np.float32)
    return preprocessed_data


def mean_shift(data, bandwidth=0.5, max_iterations=100, tol=1e-4):
    device_data = cuda.to_device(data)
    device_result = cuda.device_array_like(data)
    
    bandwith_squared = bandwidth ** 2
    
    threads_per_block = 256
    blocks_per_grid = math.ceil(data.shape[0] / threads_per_block)
    
    for iter_num in range(max_iterations):
        move_points[blocks_per_grid, threads_per_block](device_data, bandwith_squared, device_result)
        cuda.synchronize()
        current_data = device_result.copy_to_host()
        change = np.linalg.norm(current_data - data)
        if change < tol:
            break
        data = current_data
        device_data = cuda.to_device(data)
    
    return current_data

In [77]:
def test_meanshift_numba(data):
    df = pd.read_csv(f'all-data/{data}')
    data_gpu = cp.asarray(df)
    print(f'used data: {data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
    else:
        classes = None
    processed_data = preprocess_data(df)
    predicted_labels = mean_shift(processed_data)
    predicted_labels_gpu = cp.array(predicted_labels, dtype=cp.int32)
    classes_gpu =  cp.array(classes, dtype=cp.int32)
    f1_score = f1(classes_gpu, predicted_labels_gpu)
    print(f'clusters: {predicted_labels}')
    print(f'F1 score: {f1_score}')

In [86]:
# PyCuda test

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pycuda.driver as drv
import pycuda.autoinit
from pycuda.compiler import SourceModule
import math

mod = SourceModule("""
__global__ void assign_clusters(float* data, float* centroids, int* labels, int num_data, int num_centroids, int dims) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    if (idx < num_data) {
        float min_dist = INFINITY;
        int best_label = -1;
        for (int c = 0; c < num_centroids; ++c) {
            float dist = 0.0f;
            for (int i = 0; i < dims; ++i) {
                float diff = data[idx * dims + i] - centroids[c * dims + i];
                dist += diff * diff;
            }
            if (dist < min_dist) {
                min_dist = dist;
                best_label = c;
            }
        }
        labels[idx] = best_label;
    }
}

__global__ void update_centroids(float* data, int* labels, float* centroids, int num_data, int num_centroids, int dims) {
    extern __shared__ float sdata[];
    int tid = threadIdx.x;
    int bid = blockIdx.x;
    int num_threads = blockDim.x;
    
    for (int i = tid; i < dims; i += num_threads) {
        sdata[i] = 0.0f;
        __syncthreads();
        
        for (int j = 0; j < num_data; ++j) {
            if (labels[j] == bid) {
                atomicAdd(&sdata[i], data[j * dims + i]);
            }
        }
        __syncthreads();
        
        if (tid == 0) {
            int count = 0;
            for (int j = 0; j < num_data; ++j) {
                if (labels[j] == bid) {
                    count++;
                }
            }
            if (count > 0) {
                centroids[bid * dims + i] = sdata[i] / count;
            }
        }
    }
}
""")

assign_clusters_kernel = mod.get_function("assign_clusters")
update_centroids_kernel = mod.get_function("update_centroids")

def preprocess_data(df):
    """Подготовка табличных данных для K-means."""
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df)
    return scaled_data.astype(np.float32)

def kmeans(data, k, max_iterations=300, tol=1e-4):
    free_mem, total_mem = drv.mem_get_info()
    required_memory = data.nbytes + (k * data.shape[1]) * 4 + data.shape[0] * 4
    if required_memory > free_mem:
        raise MemoryError("Недостаточно памяти на GPU для размещения данных.")
    
    gpu_data = drv.mem_alloc(data.nbytes)
    drv.memcpy_htod(gpu_data, data)
    
    centroids = data[np.random.choice(data.shape[0], k, replace=False)]
    gpu_centroids = drv.mem_alloc(centroids.nbytes)
    drv.memcpy_htod(gpu_centroids, centroids)
    
    labels = np.zeros(data.shape[0], dtype=np.int32)
    gpu_labels = drv.mem_alloc(labels.nbytes)
    
    block_size = 256
    grid_size = math.ceil(data.shape[0] / block_size)
    
    for _ in range(max_iterations):
        assign_clusters_kernel(gpu_data, gpu_centroids, gpu_labels, np.int32(data.shape[0]), np.int32(k), np.int32(data.shape[1]), block=(block_size, 1, 1), grid=(grid_size, 1))
        
        update_centroids_kernel(gpu_data, gpu_labels, gpu_centroids, np.int32(data.shape[0]), np.int32(k), np.int32(data.shape[1]), block=(block_size, 1, 1), grid=(grid_size, 1), shared=block_size * 4)
        
        drv.memcpy_dtoh(labels, gpu_labels)
        drv.memcpy_dtoh(centroids, gpu_centroids)
        
        changes = np.linalg.norm(centroids - gpu_centroids)
        if changes < tol:
            break
    
    return labels, centroids


In [7]:
def test_kmeans_pycuda(data):
    df = pd.read_csv(f'all-data/{data}')
    print(f'used data: {data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
    else:
        classes = None
    processed_data = preprocess_data(df)
    print(processed_data.nbytes)
    predicted_labels, centroids = kmeans(processed_data, k=8)
    predicted_labels_gpu = cp.array(predicted_labels, dtype=cp.int32)
    classes_gpu =  cp.array(classes, dtype=cp.int32)
    f1_score = f1(classes_gpu, predicted_labels_gpu)
    print(f'clusters: {predicted_labels}')
    print(f'F1 score: {f1_score}')

In [8]:
start = time.time()
test_kmeans_pycuda('OpenML-har-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
23152152


LogicError: cuMemcpyDtoH failed: an illegal memory access was encountered

In [9]:
import numpy as np
import pycuda.autoinit
from pycuda.compiler import SourceModule
from pycuda import gpuarray

code = """
    #include <curand_kernel.h>

    const int nstates = %(NGENERATORS)s;
    __device__ curandState_t* states[nstates];

    __global__ void initkernel(int seed)
    {
        int tidx = threadIdx.x + blockIdx.x * blockDim.x;

        if (tidx < nstates) {
            curandState_t* s = new curandState_t;
            if (s != 0) {
                curand_init(seed, tidx, 0, s);
            }

            states[tidx] = s;
        }
    }

    __global__ void randfillkernel(float *values, int N)
    {
        int tidx = threadIdx.x + blockIdx.x * blockDim.x;

        if (tidx < nstates) {
            curandState_t s = *states[tidx];
            for(int i=tidx; i < N; i += blockDim.x * gridDim.x) {
                values[i] = curand_uniform(&s);
            }
            *states[tidx] = s;
        }
    }
"""

N = 1024
mod = SourceModule(code % { "NGENERATORS" : N }, no_extern_c=True, arch="sm_89")
init_func = mod.get_function("_Z10initkerneli")
fill_func = mod.get_function("_Z14randfillkernelPfi")

seed = np.int32(123456789)
nvalues = 10 * N
init_func(seed, block=(N,1,1), grid=(1,1,1))
gdata = gpuarray.zeros(nvalues, dtype=np.float32)
fill_func(gdata, np.int32(nvalues), block=(N,1,1), grid=(1,1,1))

LogicError: cuModuleLoadDataEx failed: an illegal memory access was encountered - 

In [10]:
import pycuda.autoinit
import pycuda.driver as drv
import numpy

from pycuda.compiler import SourceModule
mod = SourceModule("""
__global__ void multiply_them(float *dest, float *a, float *b)
{
  const int i = threadIdx.x;
  dest[i] = a[i] * b[i];
}
""")

multiply_them = mod.get_function("multiply_them")

a = numpy.random.randn(400).astype(numpy.float32)
b = numpy.random.randn(400).astype(numpy.float32)

dest = numpy.zeros_like(a)
multiply_them(
        drv.Out(dest), drv.In(a), drv.In(b),
        block=(400,1,1), grid=(1,1))

print(dest-a*b)

LogicError: cuModuleLoadDataEx failed: an illegal memory access was encountered - 

In [12]:
# PyCuda не работает в текущей сборке GaMAC, ни алгоритм, ни простое выделение памяти, ни пример из документации не запускаются
# документация: https://documen.tician.de/pycuda/
# Требуется сборка отдельного образа: https://forums.developer.nvidia.com/t/cant-install-pycuda/238230/6, что нецелесообразно в связи с текущей версией библиотеки на cuPY (используется базовый образ GaMAC)